In [56]:
import pandas as pd

In [57]:
df=pd.read_csv("surat_house_price.csv")
df.head(2)

,property_name,areaWithType,square_feet,transaction,status,floor,furnishing,facing,description,price_per_sqft,price
0,2 BHK Apartment for Sale in Dindoli Surat,Carpet Area,644 sqft,New Property,Poss. by Oct '24,5 out of 10,Unfurnished,West,"Luxury project with basement parking, Solar ro...","₹2,891 per sqft",₹33.8 Lac
1,2 BHK Apartment for Sale in Althan Surat,Super Area,1278 sqft,New Property,Poss. by Jan '26,6 out of 14,Unfurnished,South -West,2 And 3 BHK Luxurious Flat for Sell In New Alt...,"₹3,551 per sqft",₹45.4 Lac


In [58]:
df.dtypes

property_name     object
areaWithType      object
square_feet       object
transaction       object
status            object
floor             object
furnishing        object
facing            object
description       object
price_per_sqft    object
price             object
dtype: object

In [59]:
df.isnull().sum()

property_name        0
areaWithType         0
square_feet          0
transaction        104
status               1
floor               45
furnishing         340
facing             589
description       1371
price_per_sqft     368
price                0
dtype: int64

In [64]:
df = df.drop_duplicates()

In [65]:
df.duplicated().sum()

0

In [66]:
df = df.rename(columns={"areaWithType": "area_with_type"})

In [67]:
df["square_feet"] = (
    df["square_feet"]
    .str.replace("sqft", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["square_feet"] = pd.to_numeric(
    df["square_feet"],
    errors="coerce"
)

df["square_feet"]


0        644.0
1       1278.0
2       1173.0
3        700.0
4       1250.0
         ...  
4520    2000.0
4521    3600.0
4522    2250.0
4523    3450.0
4524    4500.0
Name: square_feet, Length: 4416, dtype: float64

In [68]:
df["area_with_type"] = df["area_with_type"].str.strip().str.title()

# Check unique values
print(df["area_with_type"].unique())

['Carpet Area' 'Super Area' 'Plot Area' 'Built Area' 'Transaction'
 'Status']


In [69]:
# Rename column
df = df.rename(columns={"area_with_type": "area_type"})

# Standardize area type values
area_type_mapping = {
    "Carpet Area": "Carpet",
    "Super Area": "Super",
    "Plot Area": "Plot",
    "Built Area": "Built-up",
    "Transaction": "Transaction",
    "Status": "Status"
}

df["area_type"] = df["area_type"].replace(area_type_mapping)

# Check result
print(df["area_type"].unique())

['Carpet' 'Super' 'Plot' 'Built-up' 'Transaction' 'Status']


In [16]:
df.columns

Index(['property_name', 'area_type', 'square_feet', 'transaction', 'status',
       'floor', 'furnishing', 'facing', 'description', 'price_per_sqft',
       'price'],
      dtype='object')

In [70]:
#property name
df["property_name"] = (
    df["property_name"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [71]:
df["transaction"] = (
    df["transaction"]
    .astype(str)
    .str.strip()
    .str.title()
)
df["transaction"].unique()

array(['New Property', 'Resale', 'Orchid Fantasia', 'Unfurnished',
       'Orchid Gardenia', 'Congo', 'Furnished', 'Nan', 'Semi-Furnished',
       'Main Road, Garden/Park', '1', '3', '4', 'Yes', 'No',
       'Mahavir Nagar', '3 Covered', 'Shagun Residency', '2', 'Main Road',
       '10 M', 'Other', '1 Covered', 'Soham Elegance', 'Garden/Park',
       'Aakash Echo Point', 'Sundaram Residency', '5', 'Jolly Residency',
       'Mangalam Heights', 'Green City', 'Pool, Garden/Park, Main Road',
       '6', 'Co-Operative Society', 'Vatsalya Bungalows', '20 Covered,',
       'Raghuvir Silverstone', '8 M', 'Casa Rivera'], dtype=object)

In [24]:
#status
df["status"] = (
    df["status"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [72]:
#floor
df["floor"] = (
    df["floor"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [73]:
#price per quare feet

df["price_per_sqft"] = df["price_per_sqft"].str.replace("₹", "")
df["price_per_sqft"] = df["price_per_sqft"].str.replace("per sqft", "", case=False)
df["price_per_sqft"] = df["price_per_sqft"].str.replace(",", "").str.strip()
df["price_per_sqft"] = pd.to_numeric(df["price_per_sqft"], errors="coerce")


In [74]:
import numpy as np

In [75]:
def price_strd(val):
    if pd.isna(val):
        return np.nan
    val = str(val).replace("₹", "").replace(",", "").strip()
    if "Cr" in val:
        return float(val.replace("Cr", "").strip()) * 100  # Convert Cr to Lacs
    elif "Lac" in val:
        return float(val.replace("Lac", "").strip())
    return np.nan

df["price_in_lacs"] = df["price"].apply(price_strd)


In [30]:
# Extract BHK number (e.g., '2 BHK' -> 2.0)
df["bhk"] = df["property_name"].str.extract(r'(\d+)\s*BHK').astype(float)

# Extract Area Location (e.g., 'in Dindoli Surat' -> 'Dindoli')
df["location"] = df["property_name"].str.extract(r'for\s+Sale\s+in\s+(.*?)\s+Surat')[0].str.strip()


In [76]:
# Extract floorno: curr and total
df["current_floor"] = df["floor"].str.extract(r'(\d+)\s+out\s+of').astype(float)
df["total_floors"] = df["floor"].str.extract(r'out\s+of\s+(\d+)').astype(float)


In [77]:
#  missing furnishing
df["furnishing"] = df["furnishing"].fillna(df["furnishing"].mode()[0])

In [80]:
# Fill missing price_per_sqft :
# (price_in_lacs × 100,000) / square_feet

missing_mask = (
    df["price_per_sqft"].isna()
    & df["square_feet"].notna()
    & df["price_in_lacs"].notna()
    & (df["price_per_sqft"] > 0)
)

df.loc[missing_mask, "price_per_sqft"] = (
    df.loc[missing_mask, "price_in_lacs"] * 100000
    / df.loc[missing_mask, "square_feet"]
)

property_name      object
area_type          object
square_feet       float64
transaction        object
status             object
floor              object
furnishing         object
facing             object
description        object
price_per_sqft    float64
price              object
price_in_lacs     float64
current_floor     float64
total_floors      float64
dtype: object

In [88]:
df.isnull().sum()

property_name       0
area_type           0
square_feet         0
transaction         0
status              0
floor               0
furnishing          0
facing              0
description         0
price_per_sqft    358
price               0
price_in_lacs     172
current_floor       0
total_floors        0
dtype: int64

In [83]:
df["facing"] = df["facing"].fillna("Unknown")
df["description"] = df["description"].fillna("No description")


In [84]:
df.isnull().sum()

property_name        0
area_type            0
square_feet        188
transaction          0
status               1
floor                0
furnishing           0
facing               0
description          0
price_per_sqft     358
price                0
price_in_lacs      172
current_floor     1248
total_floors       855
dtype: int64

In [87]:
df["square_feet"] = df["square_feet"].fillna(df["square_feet"].median())

df["status"] = df["status"].fillna("Unknown")



df["current_floor"] = df["current_floor"].fillna("Unknown")

df["total_floors"] = df["total_floors"].fillna("Unknown")

In [92]:
df.isnull().sum()

property_name     0
area_type         0
square_feet       0
transaction       0
status            0
floor             0
furnishing        0
facing            0
description       0
price_per_sqft    0
price             0
price_in_lacs     0
current_floor     0
total_floors      0
dtype: int64

In [90]:
df.dtypes

property_name      object
area_type          object
square_feet       float64
transaction        object
status             object
floor              object
furnishing         object
facing             object
description        object
price_per_sqft    float64
price              object
price_in_lacs     float64
current_floor      object
total_floors       object
dtype: object

In [91]:
# Numerical columns → median
df["price_per_sqft"] = df["price_per_sqft"].fillna(
    df["price_per_sqft"].median()
)

df["price_in_lacs"] = df["price_in_lacs"].fillna(
    df["price_in_lacs"].median()
)

# Object/string column → mode
df["price"] = df["price"].fillna(
    df["price"].mode()[0]
)